# Neural Networks from Scratch — Cumulative Code Notebook

This notebook builds the **entire neural network library** incrementally, one lecture at a time.
Each section adds new code on top of the previous sections.
By the end, you'll have a fully working neural network trained on spiral data.

**Run cells in order from top to bottom.** Each section corresponds to one or more blog posts.

---
## Setup

In [ ]:
import numpy as np
np.random.seed(0)

# If you have nnfs installed:
# pip install nnfs
try:
    import nnfs
    from nnfs.datasets import spiral_data
    nnfs.init()
except ImportError:
    # Simple spiral_data implementation if nnfs is not installed
    def spiral_data(samples, classes):
        X = np.zeros((samples * classes, 2))
        y = np.zeros(samples * classes, dtype='uint8')
        for class_number in range(classes):
            ix = range(samples * class_number, samples * (class_number + 1))
            r = np.linspace(0.0, 1, samples)
            t = np.linspace(class_number * 4, (class_number + 1) * 4, samples) + np.random.randn(samples) * 0.2
            X[ix] = np.c_[r * np.sin(t * 2.5), r * np.cos(t * 2.5)]
            y[ix] = class_number
        return X, y

print("Setup complete!")

---
## Parts 1–3: Neurons, Dot Products & Stacking Layers

A single neuron computes: $z = \sum w_i x_i + b$

A layer of neurons does this in parallel using matrix multiplication.

In [ ]:
# Part 1: A single neuron
inputs = [1.0, 2.0, 3.0, 2.5]
weights = [0.2, 0.8, -0.5, 1.0]
bias = 2.0

output = sum(w * x for w, x in zip(weights, inputs)) + bias
print(f"Single neuron output: {output}")

# Part 2: Using NumPy
output_np = np.dot(weights, inputs) + bias
print(f"NumPy dot product:   {output_np}")

# Part 3: A layer with batch processing
X = np.array([[1.0, 2.0, 3.0, 2.5],
              [2.0, 5.0, -1.0, 2.0],
              [-1.5, 2.7, 3.3, -0.8]])

W = np.array([[0.2, 0.8, -0.5, 1.0],
              [0.5, -0.91, 0.26, -0.5],
              [-0.26, -0.27, 0.17, 0.87]]).T  # (4, 3)

b = np.array([[2.0, 3.0, 0.5]])

layer_output = np.dot(X, W) + b
print(f"\nLayer output shape: {layer_output.shape}")
print(layer_output)

---
## Part 4: Dense Layer Class & Spiral Data

In [ ]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons,
                 weight_regularizer_l1=0, weight_regularizer_l2=0,
                 bias_regularizer_l1=0, bias_regularizer_l2=0):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
        # Regularization strengths
        self.weight_regularizer_l1 = weight_regularizer_l1
        self.weight_regularizer_l2 = weight_regularizer_l2
        self.bias_regularizer_l1 = bias_regularizer_l1
        self.bias_regularizer_l2 = bias_regularizer_l2

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):
        # Gradients on parameters
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        # L1 regularization gradients
        if self.weight_regularizer_l1 > 0:
            dL1 = np.ones_like(self.weights)
            dL1[self.weights < 0] = -1
            self.dweights += self.weight_regularizer_l1 * dL1
        if self.bias_regularizer_l1 > 0:
            dL1 = np.ones_like(self.biases)
            dL1[self.biases < 0] = -1
            self.dbiases += self.bias_regularizer_l1 * dL1
        # L2 regularization gradients
        if self.weight_regularizer_l2 > 0:
            self.dweights += 2 * self.weight_regularizer_l2 * self.weights
        if self.bias_regularizer_l2 > 0:
            self.dbiases += 2 * self.bias_regularizer_l2 * self.biases
        # Gradient on inputs (for previous layer)
        self.dinputs = np.dot(dvalues, self.weights.T)

# Create spiral dataset
X, y = spiral_data(samples=100, classes=3)
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Classes: {np.unique(y)}")

# Test the layer
dense_test = Layer_Dense(2, 3)
dense_test.forward(X)
print(f"Layer output shape: {dense_test.output.shape}")
print(f"First 5 outputs:\n{dense_test.output[:5]}")

---
## Part 6: Activation Functions — ReLU & Softmax

In [ ]:
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0


class Activation_Softmax:
    def forward(self, inputs):
        self.inputs = inputs
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        self.output = exp_values / np.sum(exp_values, axis=1, keepdims=True)

    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)
        for index, (single_output, single_dvalues) in \
                enumerate(zip(self.output, dvalues)):
            single_output = single_output.reshape(-1, 1)
            jacobian_matrix = np.diagflat(single_output) - \
                np.dot(single_output, single_output.T)
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalues)


# Test ReLU
relu = Activation_ReLU()
test_input = np.array([[-1, 2, -0.5, 3, 0]])
relu.forward(test_input)
print(f"ReLU input:  {test_input}")
print(f"ReLU output: {relu.output}")

# Test Softmax
softmax = Activation_Softmax()
test_logits = np.array([[2.0, 1.0, 0.1]])
softmax.forward(test_logits)
print(f"\nSoftmax input:  {test_logits}")
print(f"Softmax output: {softmax.output}")
print(f"Sum: {softmax.output.sum():.6f}")

---
## Parts 7–8: Complete Forward Pass & Loss

In [ ]:
class Loss:
    def calculate(self, output, y):
        sample_losses = self.forward(output, y)
        return np.mean(sample_losses)

    def regularization_loss(self, layer):
        regularization_loss = 0
        if layer.weight_regularizer_l1 > 0:
            regularization_loss += layer.weight_regularizer_l1 * np.sum(np.abs(layer.weights))
        if layer.weight_regularizer_l2 > 0:
            regularization_loss += layer.weight_regularizer_l2 * np.sum(layer.weights ** 2)
        if layer.bias_regularizer_l1 > 0:
            regularization_loss += layer.bias_regularizer_l1 * np.sum(np.abs(layer.biases))
        if layer.bias_regularizer_l2 > 0:
            regularization_loss += layer.bias_regularizer_l2 * np.sum(layer.biases ** 2)
        return regularization_loss


class Loss_CategoricalCrossentropy(Loss):
    def forward(self, y_pred, y_true):
        samples = len(y_pred)
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)
        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)
        return -np.log(correct_confidences)

    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        labels = len(dvalues[0])
        if len(y_true.shape) == 1:
            y_true = np.eye(labels)[y_true]
        self.dinputs = -y_true / dvalues
        self.dinputs = self.dinputs / samples


# Run the full forward pass
dense1 = Layer_Dense(2, 64)
activation1 = Activation_ReLU()
dense2 = Layer_Dense(64, 3)
activation2 = Activation_Softmax()
loss_function = Loss_CategoricalCrossentropy()

dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
activation2.forward(dense2.output)

print(f"Softmax output (first 5):\n{activation2.output[:5]}")
print(f"Row sums: {activation2.output[:5].sum(axis=1)}")

loss = loss_function.calculate(activation2.output, y)
predictions = np.argmax(activation2.output, axis=1)
accuracy = np.mean(predictions == y)

print(f"\nLoss: {loss:.4f} (expected ~1.099 for random)")
print(f"Accuracy: {accuracy:.4f} (expected ~0.33 for random)")

---
## Parts 18–19: Combined Softmax + Cross-Entropy (Backward Shortcut)

In [ ]:
class Activation_Softmax_Loss_CategoricalCrossentropy:
    """Combined Softmax activation + Cross-Entropy loss for faster, more
    numerically stable backward pass."""

    def __init__(self):
        self.activation = Activation_Softmax()
        self.loss = Loss_CategoricalCrossentropy()

    def forward(self, inputs, y_true):
        self.activation.forward(inputs)
        self.output = self.activation.output
        return self.loss.calculate(self.output, y_true)

    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis=1)
        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs /= samples


# Test the combined class
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
loss = loss_activation.forward(dense2.output, y)

print(f"Loss (combined): {loss:.4f}")

# Backward pass
loss_activation.backward(loss_activation.output, y)
print(f"Gradient shape: {loss_activation.dinputs.shape}")
print(f"Gradient sample:\n{loss_activation.dinputs[:3]}")

---
## Parts 12–21: Full Backpropagation

Now we chain the backward pass through every layer:

In [ ]:
# Build fresh network
np.random.seed(0)
X, y = spiral_data(samples=100, classes=3)

dense1 = Layer_Dense(2, 64)
activation1 = Activation_ReLU()
dense2 = Layer_Dense(64, 3)
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

# Forward
dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
loss = loss_activation.forward(dense2.output, y)

print(f"Initial loss: {loss:.4f}")

# Backward
loss_activation.backward(loss_activation.output, y)
dense2.backward(loss_activation.dinputs)
activation1.backward(dense2.dinputs)
dense1.backward(activation1.dinputs)

# Check gradient shapes
print(f"\nGradient shapes:")
print(f"  dense1.dweights: {dense1.dweights.shape} (matches weights: {dense1.weights.shape})")
print(f"  dense1.dbiases:  {dense1.dbiases.shape}  (matches biases:  {dense1.biases.shape})")
print(f"  dense2.dweights: {dense2.dweights.shape} (matches weights: {dense2.weights.shape})")
print(f"  dense2.dbiases:  {dense2.dbiases.shape}  (matches biases:  {dense2.biases.shape})")

# One manual update step
lr = 0.01
dense1.weights -= lr * dense1.dweights
dense1.biases -= lr * dense1.dbiases
dense2.weights -= lr * dense2.dweights
dense2.biases -= lr * dense2.dbiases

# Check if loss decreased
dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
new_loss = loss_activation.forward(dense2.output, y)
print(f"\nLoss after 1 update: {new_loss:.4f} (decreased by {loss - new_loss:.4f})")

---
## Parts 22–27: Optimizers

We implement all 6 optimizer variants. Each one builds on the previous.

In [ ]:
class Optimizer_SGD:
    def __init__(self, learning_rate=1.0, decay=0., momentum=0.):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.momentum = momentum

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate / \
                (1. + self.decay * self.iterations)

    def update_params(self, layer):
        if self.momentum:
            if not hasattr(layer, 'weight_momentums'):
                layer.weight_momentums = np.zeros_like(layer.weights)
                layer.bias_momentums = np.zeros_like(layer.biases)
            weight_updates = self.momentum * layer.weight_momentums - \
                self.current_learning_rate * layer.dweights
            layer.weight_momentums = weight_updates
            bias_updates = self.momentum * layer.bias_momentums - \
                self.current_learning_rate * layer.dbiases
            layer.bias_momentums = bias_updates
        else:
            weight_updates = -self.current_learning_rate * layer.dweights
            bias_updates = -self.current_learning_rate * layer.dbiases
        layer.weights += weight_updates
        layer.biases += bias_updates

    def post_update_params(self):
        self.iterations += 1


class Optimizer_Adagrad:
    def __init__(self, learning_rate=1.0, decay=0., epsilon=1e-7):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate / \
                (1. + self.decay * self.iterations)

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)
        layer.weight_cache += layer.dweights ** 2
        layer.bias_cache += layer.dbiases ** 2
        layer.weights -= self.current_learning_rate * layer.dweights / \
            (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases -= self.current_learning_rate * layer.dbiases / \
            (np.sqrt(layer.bias_cache) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


class Optimizer_RMSprop:
    def __init__(self, learning_rate=0.001, decay=0., epsilon=1e-7, rho=0.9):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.rho = rho

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate / \
                (1. + self.decay * self.iterations)

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)
        layer.weight_cache = self.rho * layer.weight_cache + \
            (1 - self.rho) * layer.dweights ** 2
        layer.bias_cache = self.rho * layer.bias_cache + \
            (1 - self.rho) * layer.dbiases ** 2
        layer.weights -= self.current_learning_rate * layer.dweights / \
            (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases -= self.current_learning_rate * layer.dbiases / \
            (np.sqrt(layer.bias_cache) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


class Optimizer_Adam:
    def __init__(self, learning_rate=0.001, decay=0., epsilon=1e-7,
                 beta_1=0.9, beta_2=0.999):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate / \
                (1. + self.decay * self.iterations)

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.biases)
            layer.bias_cache = np.zeros_like(layer.biases)

        # Update momentum
        layer.weight_momentums = self.beta_1 * layer.weight_momentums + \
            (1 - self.beta_1) * layer.dweights
        layer.bias_momentums = self.beta_1 * layer.bias_momentums + \
            (1 - self.beta_1) * layer.dbiases

        # Bias-corrected momentum
        weight_momentums_corrected = layer.weight_momentums / \
            (1 - self.beta_1 ** (self.iterations + 1))
        bias_momentums_corrected = layer.bias_momentums / \
            (1 - self.beta_1 ** (self.iterations + 1))

        # Update cache
        layer.weight_cache = self.beta_2 * layer.weight_cache + \
            (1 - self.beta_2) * layer.dweights ** 2
        layer.bias_cache = self.beta_2 * layer.bias_cache + \
            (1 - self.beta_2) * layer.dbiases ** 2

        # Bias-corrected cache
        weight_cache_corrected = layer.weight_cache / \
            (1 - self.beta_2 ** (self.iterations + 1))
        bias_cache_corrected = layer.bias_cache / \
            (1 - self.beta_2 ** (self.iterations + 1))

        # Update parameters
        layer.weights -= self.current_learning_rate * \
            weight_momentums_corrected / \
            (np.sqrt(weight_cache_corrected) + self.epsilon)
        layer.biases -= self.current_learning_rate * \
            bias_momentums_corrected / \
            (np.sqrt(bias_cache_corrected) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


print("All 4 optimizer classes defined:")
print("  - Optimizer_SGD (with optional decay & momentum)")
print("  - Optimizer_Adagrad")
print("  - Optimizer_RMSprop")
print("  - Optimizer_Adam")

---
## Part 31: Dropout Layer

In [ ]:
class Layer_Dropout:
    def __init__(self, rate):
        # Store the SUCCESS rate (1 - dropout_rate)
        self.rate = 1 - rate

    def forward(self, inputs, training=True):
        self.inputs = inputs
        if not training:
            self.output = inputs.copy()
            return
        # Inverted dropout: scale by 1/rate during training
        self.binary_mask = np.random.binomial(1, self.rate,
                           size=inputs.shape) / self.rate
        self.output = inputs * self.binary_mask

    def backward(self, dvalues):
        self.dinputs = dvalues * self.binary_mask


# Test dropout
dropout_test = Layer_Dropout(0.2)
test_data = np.ones((1, 10))
dropout_test.forward(test_data, training=True)
print(f"Dropout (rate=0.2) output: {dropout_test.output}")
print(f"Fraction zeroed: {np.mean(dropout_test.output == 0):.2f}")

---
## Complete Training Loop — Putting It All Together

Now we combine everything into a full training loop using Adam optimizer with L2 regularization and dropout.

In [ ]:
# Fresh dataset and network
np.random.seed(0)
X, y = spiral_data(samples=100, classes=3)

# Network architecture
dense1 = Layer_Dense(2, 64, weight_regularizer_l2=5e-4)
activation1 = Activation_ReLU()
dropout1 = Layer_Dropout(0.1)
dense2 = Layer_Dense(64, 3)
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

# Optimizer
optimizer = Optimizer_Adam(learning_rate=0.05, decay=5e-7)

# Training loop
for epoch in range(10001):
    # Forward pass
    dense1.forward(X)
    activation1.forward(dense1.output)
    dropout1.forward(activation1.output, training=True)
    dense2.forward(dropout1.output)

    # Loss
    data_loss = loss_activation.forward(dense2.output, y)
    reg_loss = loss_activation.loss.regularization_loss(dense1) + \
               loss_activation.loss.regularization_loss(dense2)
    loss = data_loss + reg_loss

    # Accuracy
    predictions = np.argmax(loss_activation.output, axis=1)
    accuracy = np.mean(predictions == y)

    # Print progress
    if epoch % 1000 == 0:
        print(f"epoch: {epoch:5d} | loss: {loss:.4f} (data: {data_loss:.4f} + reg: {reg_loss:.4f}) | acc: {accuracy:.4f} | lr: {optimizer.current_learning_rate:.5f}")

    # Backward pass
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    dropout1.backward(dense2.dinputs)
    activation1.backward(dropout1.dinputs)
    dense1.backward(activation1.dinputs)

    # Update weights
    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

---
## Validation on Test Data

In [ ]:
# Generate test data with different seed
X_test, y_test = spiral_data(samples=100, classes=3)

# Forward pass with dropout DISABLED
dense1.forward(X_test)
activation1.forward(dense1.output)
dropout1.forward(activation1.output, training=False)
dense2.forward(dropout1.output)

test_loss = loss_activation.forward(dense2.output, y_test)
test_predictions = np.argmax(loss_activation.output, axis=1)
test_accuracy = np.mean(test_predictions == y_test)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"\nTrain accuracy: {accuracy:.4f}")
print(f"Generalization gap: {accuracy - test_accuracy:.4f}")

---
## Summary

**Everything above was built from scratch using only NumPy.** Here's what we implemented:

| Component | Class | Blog Parts |
|---|---|---|
| Dense Layer | `Layer_Dense` | 1–5 |
| ReLU Activation | `Activation_ReLU` | 6 |
| Softmax Activation | `Activation_Softmax` | 6 |
| Cross-Entropy Loss | `Loss_CategoricalCrossentropy` | 8 |
| Combined Softmax+CCE | `Activation_Softmax_Loss_CategoricalCrossentropy` | 18–19 |
| SGD Optimizer (+ decay + momentum) | `Optimizer_SGD` | 22–24 |
| AdaGrad Optimizer | `Optimizer_Adagrad` | 25 |
| RMSProp Optimizer | `Optimizer_RMSprop` | 26 |
| Adam Optimizer | `Optimizer_Adam` | 27 |
| Dropout Layer | `Layer_Dropout` | 31 |
| L1/L2 Regularization | (built into `Layer_Dense`) | 30 |

**Next step:** Apply this to a real dataset! See `projects/01-mnist-from-scratch/README.md` for the MNIST build.